In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("datasnaek/youtube-new")

print("Path to dataset files:", path)

In [ ]:
import os
os.listdir(path)
file_path = os.path.join(path, "USvideos.csv")
print("File path:", file_path)


In [ ]:
data=pd.read_csv(file_path)
data.head()

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
data.dtypes

In [ ]:
data.shape

In [ ]:
data.columns

In [ ]:
clean_data=data.copy()

In [ ]:
clean_data.columns-clean_data.columns.str.strip().str.lower().str.replace(" ","_")  

In [ ]:
clean_data.isnull().sum()

In [ ]:
missing_percent=clean_data.isnull().sum()/len(clean_data)*100
missing_percent.sort_values(ascending=False)

In [ ]:
clean_data['description']=clean_data['description'].fillna("No description")    

In [ ]:
clean_data.dropna(subset=['video_id','title'],inplace=True)

In [ ]:
clean_data.duplicated().sum()

In [ ]:
clean_data=clean_data.drop_duplicates()

In [ ]:
clean_data=clean_data.reset_index(drop=True)
clean_data.dtypes

In [ ]:
clean_data['publish_time']=pd.to_datetime(clean_data['publish_time'],errors='coerce')
clean_data['trending_date']=pd.to_datetime(clean_data['trending_date'],format='%y.%d.%m',errors='coerce ') 

In [ ]:
clean_data=clean_data[clean_data['likes']>=0]

In [ ]:
clean_data=clean_data[clean_data['comment_count']>=0]

In [ ]:
text_columns=['title','channel_title']
for col in text_columns:
    clean_data[col]=clean_data[col].str.strip()


In [ ]:
plt.figure(figsize=(10,6))
sns.boxplot(x=clean_data['views'])
plt.show()

In [ ]:
Q1=clean_data['views'].quantile(0.25)
Q3=clean_data['views'].quantile(0.75)
IQR=Q3-Q1
lower_bound=Q1-1.5*IQR
upper_bound=Q3+1.5*IQR
clean_data=clean_data[(clean_data['views']>=lower_bound) & (clean_data['views']<=upper_bound)]  

In [ ]:
plt.figure(figsize=(10,6))
sns.boxplot(x=clean_data['views'])
plt.show()

In [ ]:
clean_data['engagement']=((clean_data['likes']+clean_data['dislikes']+clean_data['comment_count'])/clean_data['views'])*100

In [ ]:
clean_data['like_ratio']=(clean_data['likes']/(clean_data['views']))*100 

In [ ]:
clean_data['publish_hour']=clean_data['publish_time'].dt.hour

In [ ]:
clean_data['title_length']=clean_data['title'].str.len()

In [ ]:
correlation=clean_data[['views','likes','dislikes','comment_count']].corr()
correlation

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(correlation, annot=True, cmap='coolwarm', linewidths=0.5)
plt.show()

In [ ]:
clean_data.isnull().sum()

In [ ]:
clean_data.duplicated().sum()

In [ ]:
(clean_data[['views','likes','comment_count']]<0).sum()

In [ ]:
clean_data.to_csv("cleaned_youtube_data.csv", index=False)

In [ ]:
report={
    "Original Rows": len(data),
    "Cleaned Rows": len(clean_data),
    "Rows Removed": len(data)-len(clean_data),
    "Duplicate Rows Removed": clean_data.duplicated().sum(),
    "Missing Values Remaining": clean_data.isnull().sum().sum()

}
print(report)